# 198. Agent 高风险工具：动作前审批与 Policy Gate 怎样实现？

> **面试问题：如何把审批绑定到精确工具参数、策略版本和有效期，并阻断过期、重放、参数替换与提示注入后的越权调用？**

## 先给结论

不要把 Agent 面试题答成框架 API：先定义状态、动作、权限、预算、版本和可判定的终态，再讨论 prompt、模型和并发扩展。下面用受控内存数据手写最小协议；小规模断言只证明实现合同，不代表线上模型效果、权限体系或安全等级。

## 一手资料

- [AgentDojo](https://arxiv.org/abs/2406.13352)
- [Open Agent Passport](https://arxiv.org/abs/2603.20953)
- [MCP Tools](https://modelcontextprotocol.io/specification/draft/server/tools)

In [ ]:
notebook_contract = {"mode": "in-memory-demo", "oracle": "assertions", "production": "isolation-and-audit"}  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["mode"] == "in-memory-demo"  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["oracle"] == "assertions"  # 执行本行的状态、计算或校验逻辑。
assert "audit" in notebook_contract["production"]  # 执行本行的状态、计算或校验逻辑。
assert len(notebook_contract) == 3  # 执行本行的状态、计算或校验逻辑。


## 1. 问题拆解：高风险动作需要动作前、参数绑定的审批

仅在系统提示词写“敏感操作先询问用户”不够。审批 token 必须绑定精确工具名、参数摘要、主体、策略版本和有效期；任一字段改变都需要重新审批。这个机制应位于工具执行边界，而非模型文本输出之后。


In [ ]:
from dataclasses import dataclass  # 执行本行的状态、计算或校验逻辑。
@dataclass(frozen=True)  # 执行本行的状态、计算或校验逻辑。
class ToolRequest:  # 执行本行的状态、计算或校验逻辑。
    request_id: str  # 执行本行的状态、计算或校验逻辑。
    user: str  # 执行本行的状态、计算或校验逻辑。
    tool: str  # 执行本行的状态、计算或校验逻辑。
    arguments: tuple  # 执行本行的状态、计算或校验逻辑。
    policy_version: str  # 执行本行的状态、计算或校验逻辑。
request = ToolRequest("r-1", "alice", "payment.charge", (("amount", "120"),), "policy-v1")  # 执行本行的状态、计算或校验逻辑。
assert request.tool == "payment.charge"  # 执行本行的状态、计算或校验逻辑。
assert request.user == "alice"  # 执行本行的状态、计算或校验逻辑。
assert request.policy_version == "policy-v1"  # 执行本行的状态、计算或校验逻辑。


## 2. 风险分级：先区分 read、write 与不可逆动作

策略应当在模型之外由确定规则或经过审核的策略引擎执行。演示按工具前缀分类；生产中还要将金额、资源所有权、数据级别、地区、操作频率与当前会话风险加入条件。


In [ ]:
def risk_level(request):  # 执行本行的状态、计算或校验逻辑。
    if request.tool.startswith("payment."):  # 执行本行的状态、计算或校验逻辑。
        return "high"  # 执行本行的状态、计算或校验逻辑。
    if request.tool.endswith(".write"):  # 执行本行的状态、计算或校验逻辑。
        return "medium"  # 执行本行的状态、计算或校验逻辑。
    return "low"  # 执行本行的状态、计算或校验逻辑。
assert risk_level(request) == "high"  # 执行本行的状态、计算或校验逻辑。
assert risk_level(ToolRequest("r-2", "alice", "orders.read", (), "policy-v1")) == "low"  # 执行本行的状态、计算或校验逻辑。
assert risk_level(ToolRequest("r-3", "alice", "profile.write", (), "policy-v1")) == "medium"  # 执行本行的状态、计算或校验逻辑。


## 3. 请求摘要：审批必须覆盖精确的参数

不能只审批“可以付款”，因为金额、收款人或工具版本可能在后续 loop 中被修改。这里用稳定 JSON 摘要构造 request digest；生产审批记录还应包括租户、nonce、授权链与安全存储。


In [ ]:
import hashlib  # 执行本行的状态、计算或校验逻辑。
import json  # 执行本行的状态、计算或校验逻辑。
def request_digest(request):  # 执行本行的状态、计算或校验逻辑。
    payload = {"user": request.user, "tool": request.tool, "arguments": request.arguments, "policy": request.policy_version}  # 执行本行的状态、计算或校验逻辑。
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()  # 执行本行的状态、计算或校验逻辑。
digest = request_digest(request)  # 执行本行的状态、计算或校验逻辑。
assert len(digest) == 64  # 执行本行的状态、计算或校验逻辑。
assert digest == request_digest(request)  # 执行本行的状态、计算或校验逻辑。
assert digest != request_digest(ToolRequest("r-1", "alice", "payment.charge", (("amount", "121"),), "policy-v1"))  # 执行本行的状态、计算或校验逻辑。


## 4. 审批 token：包含摘要、过期时刻和一次性使用语义

高风险操作进入 pending 状态后，前端或人工审批系统展示结构化参数；批准生成 token。token 只适用于那个 request digest，并有明确的过期时间和使用次数，不能被新请求复用。


In [ ]:
@dataclass  # 执行本行的状态、计算或校验逻辑。
class Approval:  # 执行本行的状态、计算或校验逻辑。
    digest: str  # 执行本行的状态、计算或校验逻辑。
    expires_at: int  # 执行本行的状态、计算或校验逻辑。
    used: bool = False  # 执行本行的状态、计算或校验逻辑。
def approve(request, now, ttl):  # 执行本行的状态、计算或校验逻辑。
    return Approval(request_digest(request), now + ttl)  # 执行本行的状态、计算或校验逻辑。
approval = approve(request, now=10, ttl=5)  # 执行本行的状态、计算或校验逻辑。
assert approval.digest == digest  # 执行本行的状态、计算或校验逻辑。
assert approval.expires_at == 15  # 执行本行的状态、计算或校验逻辑。
assert approval.used is False  # 执行本行的状态、计算或校验逻辑。


## 5. 执行门禁：验证审批后才让工具产生副作用

执行器必须独立校验风险、token 摘要、策略版本和过期状态。即使模型被提示注入诱导修改参数，摘要不匹配也会阻断调用；真实实现还需要把审批验证和工具调用放在不可绕过的服务端事务中。


In [ ]:
ledger = {"charged": 0}  # 执行本行的状态、计算或校验逻辑。
def execute(request, approval, now):  # 执行本行的状态、计算或校验逻辑。
    if risk_level(request) == "high" and (approval is None or approval.used or approval.expires_at < now or approval.digest != request_digest(request)):  # 执行本行的状态、计算或校验逻辑。
        return {"ok": False, "reason": "approval_required"}  # 执行本行的状态、计算或校验逻辑。
    approval.used = True  # 执行本行的状态、计算或校验逻辑。
    ledger["charged"] += int(dict(request.arguments)["amount"])  # 执行本行的状态、计算或校验逻辑。
    return {"ok": True, "reason": "executed"}  # 执行本行的状态、计算或校验逻辑。
executed = execute(request, approval, 12)  # 执行本行的状态、计算或校验逻辑。
assert executed["ok"] is True  # 执行本行的状态、计算或校验逻辑。
assert ledger["charged"] == 120  # 执行本行的状态、计算或校验逻辑。
assert approval.used is True  # 执行本行的状态、计算或校验逻辑。


## 6. 失败分支：过期、重放和参数替换必须失败

审批的三类常见漏洞是 token 过期后仍被接受、同一 token 重放、把已审批的 token 换到另一个金额/收款人。下面逐一断言拒绝；拒绝事件也必须写入审计，而不是静默丢失。


In [ ]:
replay = execute(request, approval, 12)  # 执行本行的状态、计算或校验逻辑。
changed = ToolRequest("r-1", "alice", "payment.charge", (("amount", "121"),), "policy-v1")  # 执行本行的状态、计算或校验逻辑。
expired = approve(changed, now=0, ttl=1)  # 执行本行的状态、计算或校验逻辑。
assert replay["ok"] is False  # 执行本行的状态、计算或校验逻辑。
assert execute(changed, expired, 2)["ok"] is False  # 执行本行的状态、计算或校验逻辑。
assert ledger["charged"] == 120  # 执行本行的状态、计算或校验逻辑。


## 7. 策略变更：旧审批不得跨 policy version 使用

策略升级可能引入新阈值或新合规要求，因此 policy version 是 digest 的一部分。升级后必须重新决策和审批；生产环境还应支持紧急 deny-list 立即吊销未使用 token。


In [ ]:
new_policy_request = ToolRequest("r-1", "alice", "payment.charge", (("amount", "120"),), "policy-v2")  # 执行本行的状态、计算或校验逻辑。
new_approval = approve(new_policy_request, now=20, ttl=3)  # 执行本行的状态、计算或校验逻辑。
assert request_digest(new_policy_request) != digest  # 执行本行的状态、计算或校验逻辑。
assert execute(new_policy_request, new_approval, 21)["ok"] is True  # 执行本行的状态、计算或校验逻辑。
assert ledger["charged"] == 240  # 执行本行的状态、计算或校验逻辑。


## 8. 审计：记录决策依据，不记录不必要的敏感内容

审计记录应包含 request digest、风险、决策、策略版本、审批时间和执行结果，以支持复放与事故调查。日志要避免直接泄露卡号、密钥或用户全文；访问与保留期同样是系统合同。


In [ ]:
audit = {"request": request.request_id, "digest": digest, "risk": risk_level(request), "policy": request.policy_version, "decision": executed["reason"]}  # 执行本行的状态、计算或校验逻辑。
audit_hash = hashlib.sha256(json.dumps(audit, sort_keys=True).encode()).hexdigest()  # 执行本行的状态、计算或校验逻辑。
assert audit["risk"] == "high"  # 执行本行的状态、计算或校验逻辑。
assert audit["decision"] == "executed"  # 执行本行的状态、计算或校验逻辑。
assert len(audit_hash) == 64  # 执行本行的状态、计算或校验逻辑。


## 面试收束

回答时依次给出目标、状态合同、动作前校验、主路径、失败分支、指标、制品版本和生产替换点。可靠 Agent 不靠模型自述“完成”，而靠独立的状态 oracle、预算约束、审计和可复放 trace。
